# Exercise: 
# Implement a history logger using the ** operator 

In [1]:


# 1. define the agents 
def planner(state):
    return {**state, "plan": "retrieve policy"}

def researcher(state):
    return {**state, "docs": ["Policy 2026"]}

def summarizer(state):
    return {**state, "answer": f"Found {state['docs'][0]}"}

# 2. set up the initial state and history tracking list 
state = {"query": "Sweden leave policy"}
state_history = [state]

# 3. Run the workflow and log every transition 
agents = [planner, researcher, summarizer]


for agent in agents:
    # Generate the next state using unpacking, preserves the old state
    state = agent(state)

    # save a snapshot of the newly generated state
    state_history.append(state)

# 4. print chronoloical logs 
print ("---------------State Transition logs---------------") 
for index, snapshot in enumerate(state_history):
    if index == 0:
        print (f"Step 0: Initia Input\n   -> {snapshot}\n")
    else: 
        print (f"Step {index}: After {agents[index-1].__name__}\n.  -> {snapshot}\n")

# print state_history[i] to see only the ith step of success / failure


---------------State Transition logs---------------
Step 0: Initia Input
   -> {'query': 'Sweden leave policy'}

Step 1: After planner
.  -> {'query': 'Sweden leave policy', 'plan': 'retrieve policy'}

Step 2: After researcher
.  -> {'query': 'Sweden leave policy', 'plan': 'retrieve policy', 'docs': ['Policy 2026']}

Step 3: After summarizer
.  -> {'query': 'Sweden leave policy', 'plan': 'retrieve policy', 'docs': ['Policy 2026'], 'answer': 'Found Policy 2026'}



# Enhancement 

### Adding a step counter or  timestamp metadata along with a rollback mechanism to this process 

Here is the complete implementation showing how to add metadata tracking (timestamps/steps) and a rollback mechanism using the ** unpacking operator.
If an agent fails validation (e.g., returns bad data or missing keys), the workflow automatically discards that step and reverts back to the last known good state.


In [ ]:
from datetime import datetime
import time

# 1. Define Agent Nodes using ** unpacking
def planner(state):
    # Overwrites metadata while maintaining history cleanly
    return {
        **state, 
        "plan": "Retrieve policy",
        "metadata": {"step": state["metadata"]["step"] + 1, "timestamp": datetime.now().strftime("%X")}
    }

def researcher_faulty(state):
    print("⚠️ [Researcher] Simulated Network Error! Returning empty documents list...")
    return {
        **state, 
        "docs": [], # Invalid state triggering our validator
        "metadata": {"step": state["metadata"]["step"] + 1, "timestamp": datetime.now().strftime("%X")}
    }

def researcher_fixed(state):
    print("🔄 [Researcher] Retry successful! Fetching real documents...")
    return {
        **state, 
        "docs": ["Sweden Leave Policy 2026"],
        "metadata": {"step": state["metadata"]["step"] + 1, "timestamp": datetime.now().strftime("%X")}
    }

def summarizer(state):
    return {
        **state, 
        "answer": f"Processed: {state['docs'][0]}",
        "metadata": {"step": state["metadata"]["step"] + 1, "timestamp": datetime.now().strftime("%X")}
    }

# 2. State Validation Check
def is_state_valid(state, step_name):
    """Returns False if critical data is missing based on workflow progression."""
    if step_name == "researcher" and not state.get("docs"):
        return False
    return True

# 3. Setup Initial State and History Logger
state = {
    "query": "Sweden leave policy",
    "metadata": {"step": 0, "timestamp": datetime.now().strftime("%X")}
}
state_history = [state]

# ==========================================
# 4. Pipeline Execution with Rollback Logic
# ==========================================

# Step 1: Run Planner
state = planner(state)
state_history.append(state)

# Step 2: Run Faulty Researcher
time.sleep(1) # Visual separation for timestamps
tentative_state = researcher_faulty(state)

if is_state_valid(tentative_state, "researcher"):
    state = tentative_state
    state_history.append(state)
else:
    # ROLLBACK: Discard tentative_state, fall back to previous valid state
    print("🚨 Validation failed! Rolling back to last valid state...")
    state = state_history[-1] 
    
    # Try an alternative backup route
    time.sleep(1)
    state = researcher_fixed(state)
    state_history.append(state)

# Step 3: Run Summarizer
time.sleep(1)
state = summarizer(state)
state_history.append(state)

# ==========================================
# 5. Output Audit Logs
# ==========================================
print("\n=== SYSTEM AUDIT LOGS ===")
for index, snapshot in enumerate(state_history):
    print(f"\nSnapshot {index} | Time: {snapshot['metadata']['timestamp']} | Step Counter: {snapshot['metadata']['step']}")
    # Print clean version without metadata block for scannability
    clean_print = {k: v for k, v in snapshot.items() if k != 'metadata'}
    print(f"  -> {clean_print}")
